# Datathon Passos Mágicos — Ingestão e Qualidade dos Dados

Este notebook faz:
- carga das abas **PEDE2022 / PEDE2023 / PEDE2024** diretamente da planilha pública
- padronização de colunas
- checagem de qualidade
- geração da base consolidada para análise e modelagem

> Se preferir, você também pode salvar os CSVs em `data_raw/` e rodar offline depois.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
(PROJECT_ROOT / "data_raw").mkdir(exist_ok=True)
(PROJECT_ROOT / "data_processed").mkdir(exist_ok=True)
(PROJECT_ROOT / "outputs" / "figs").mkdir(parents=True, exist_ok=True)

sys.path.append(str(PROJECT_ROOT / "src"))

from passos_data import LoadConfig, load_all_years, summarize_quality, add_phase_order, make_long_for_trend, get_indicator_columns
from plots_passos import plot_barras_defasagem_por_ano, plot_serie_indicadores, plot_matriz_correlacao

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

## 1) Carregar e consolidar as abas

In [ ]:
cfg = LoadConfig(
    sheet_id="1td91KoeSgXrUrCVOUkLmONG9Go3LVcXpcNEw_XrL2R0",
    sheet_names=("PEDE2022", "PEDE2023", "PEDE2024"),
    cache_dir="data_raw",
    force_refresh=False,  # troque para True se quiser baixar de novo
)

df = load_all_years(cfg)
df = add_phase_order(df)

print("Shape consolidado:", df.shape)
display(df.head(3))
display(df.tail(3))

## 2) Dicionário rápido de colunas e qualidade

In [ ]:
quality = summarize_quality(df)
display(quality.head(30))

print("Colunas disponíveis:")
print(sorted(df.columns.tolist()))

## 3) Checks importantes

In [ ]:
checks = {
    "linhas_totais": len(df),
    "anos_disponiveis": sorted(df["ano_referencia"].dropna().unique().tolist()) if "ano_referencia" in df.columns else [],
    "alunos_unicos_ra": int(df["ra"].nunique()) if "ra" in df.columns else None,
    "linhas_duplicadas_ra_ano": int(df.duplicated(subset=[c for c in ["ra","ano_referencia"] if c in df.columns]).sum()) if {"ra","ano_referencia"}.issubset(df.columns) else None,
    "pct_risco_atual": float(df["risco_defasagem_atual"].dropna().mean()) if "risco_defasagem_atual" in df.columns else None,
    "pct_risco_severo": float(df["risco_defasagem_severo"].dropna().mean()) if "risco_defasagem_severo" in df.columns else None,
}
checks

## 4) Gráficos iniciais (sanidade)

In [ ]:
if "defasagem" in df.columns:
    plot_barras_defasagem_por_ano(df, outpath="outputs/figs/00_defasagem_por_ano.png")
else:
    print("Coluna 'defasagem' não disponível; pulando gráfico.")

long_df = make_long_for_trend(df)
if not long_df.empty:
    inds = [i for i in ["inde","ida","ieg","iaa","ips","ipp","ipv","ian"] if i in long_df["indicador"].unique()]
    plot_serie_indicadores(long_df, indicadores=inds, outpath="outputs/figs/00_tendencia_indicadores.png")
else:
    print("Sem indicadores em formato long.")

corr_cols = [c for c in ["inde","ida","ieg","iaa","ips","ipp","ipv","ian","defasagem"] if c in df.columns]
if len(corr_cols) >= 2:
    plot_matriz_correlacao(df, corr_cols, outpath="outputs/figs/00_correlacao_indicadores.png")

## 5) Salvar base processada

In [ ]:
out_csv = PROJECT_ROOT / "data_processed" / "pede_consolidado.csv"
df.to_csv(out_csv, index=False)
print("Arquivo salvo em:", out_csv)

## Próximo passo

Abra o notebook **01_analise_negocio_perguntas.ipynb** para responder as perguntas de negócio e montar a narrativa do case.